# Hemo Invasion PDE Demo

This tutorial shows how to run `HemoInvasion3D` in TumorTwin using the same pattern as `HGG_Demo` and `TNBC_Demo`.

We include a **mini 50-day run** for quick sanity checking.

In [ ]:
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from tumortwin.models import HemoInvasion3D
from tumortwin.preprocessing import ADC_to_cellularity
from tumortwin.solvers import TorchDiffEqSolver, TorchDiffEqSolverOptions
from tumortwin.types import CropSettings, CropTarget
from tumortwin.types.hgg_data import HGGPatientData

In [ ]:
# Choose device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Resolve paths robustly (works whether cwd is repo root or tutorials/)
cwd = Path.cwd()
repo_root = cwd if (cwd / "tumortwin").exists() else cwd.parent

patient_json = repo_root / "input_files" / "HGG_demo_001" / "HGG_demo_001.json"
if not patient_json.exists():
    raise FileNotFoundError(f"Could not find patient json: {patient_json}")

crop_settings = CropSettings(crop_to=CropTarget.ROI_ENHANCE, padding=10, visit_index=-1)
patient_data = HGGPatientData.from_file(patient_json, crop_settings=crop_settings)

print(f"Patient: {patient_data.patient}")
print(f"Visits: {len(patient_data.visits)}")
print(f"Grid shape: {patient_data.brainmask_image.array.shape}")

In [ ]:
# Build initial fields from first visit
visit0 = patient_data.visits[0]

cellularity0 = ADC_to_cellularity(
    visit0.adc_image,
    visit0.roi_enhance_image,
    visit0.roi_nonenhance_image,
)

initial_n = torch.from_numpy(cellularity0.array).float().to(device)
initial_m = torch.zeros_like(initial_n)
initial_s = torch.ones_like(initial_n)

# We use ROI-enhancing voxels as a proxy vessel mask for the demo.
# In a production setup, replace this with a proper vessel segmentation/mask.
vessel_mask = torch.from_numpy((visit0.roi_enhance_image.array > 0).astype(np.bool_)).to(device)

print("Initial tensors:")
print("  n:", tuple(initial_n.shape), f"[{initial_n.min().item():.3f}, {initial_n.max().item():.3f}]")
print("  m:", tuple(initial_m.shape), f"[{initial_m.min().item():.3f}, {initial_m.max().item():.3f}]")
print("  s:", tuple(initial_s.shape), f"[{initial_s.min().item():.3f}, {initial_s.max().item():.3f}]")
print("  vessel voxels:", int(vessel_mask.sum().item()))

In [ ]:
# Initialize HemoInvasion3D model
model = HemoInvasion3D(
    B=torch.tensor(0.05, dtype=torch.float32, device=device),
    Dn=torch.tensor(0.01, dtype=torch.float32, device=device),
    Ds=torch.tensor(0.08, dtype=torch.float32, device=device),
    k_s=torch.tensor(0.20, dtype=torch.float32, device=device),
    s_star=torch.tensor(0.10, dtype=torch.float32, device=device),
    patient_data=patient_data,
    initial_n=initial_n,
    initial_m=initial_m,
    initial_s=initial_s,
    K=torch.tensor(1.0, dtype=torch.float32, device=device),
    s_crit=torch.tensor(0.50, dtype=torch.float32, device=device),
    s_smooth=torch.tensor(0.10, dtype=torch.float32, device=device),
    s_outside=0.0,
    s_vessel=1.0,
    vessel_mask=vessel_mask,
    time_scale_days=10.0,
    poisson_iterations=24,
    require_grad=False,
    device=device,
)

solver = TorchDiffEqSolver(
    model,
    TorchDiffEqSolverOptions(
        step_size=timedelta(days=0.25),
        method="rk4",
        device=device,
        use_adjoint=False,
    ),
)

u0 = model.get_initial_state()
print("u0 shape:", tuple(u0.shape))

In [ ]:
# Mini run: 50 days (daily outputs)
mini_t0 = patient_data.visits[0].time
mini_timepoints = [mini_t0 + timedelta(days=d) for d in range(0, 51)]  # 0..50 days

times_mini, traj_mini = solver.solve(timepoints=mini_timepoints, u_initial=u0)

# Unpack fields: traj shape = (T, 3, D, H, W)
n_series = traj_mini[:, 0]
m_series = traj_mini[:, 1]
s_series = traj_mini[:, 2]

print("Mini run complete")
print("  trajectory shape:", tuple(traj_mini.shape))
print("  n finite:", bool(torch.isfinite(n_series).all()))
print("  m finite:", bool(torch.isfinite(m_series).all()))
print("  s finite:", bool(torch.isfinite(s_series).all()))

In [ ]:
# Fast sanity checks on dynamics
mass_n = n_series.sum(dim=(1, 2, 3)).detach().cpu().numpy()
mass_m = m_series.sum(dim=(1, 2, 3)).detach().cpu().numpy()
mean_s = s_series.mean(dim=(1, 2, 3)).detach().cpu().numpy()
time_days = times_mini.detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(time_days, mass_n)
axes[0].set_title("Total proliferating cells (n)")
axes[0].set_xlabel("days")
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_days, mass_m)
axes[1].set_title("Total quiescent cells (m)")
axes[1].set_xlabel("days")
axes[1].grid(True, alpha=0.3)

axes[2].plot(time_days, mean_s)
axes[2].set_title("Mean substrate (S)")
axes[2].set_xlabel("days")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

print(f"n range: [{n_series.min().item():.4f}, {n_series.max().item():.4f}]")
print(f"m range: [{m_series.min().item():.4f}, {m_series.max().item():.4f}]")
print(f"S range: [{s_series.min().item():.4f}, {s_series.max().item():.4f}]")

In [ ]:
# Visual check: center slice for n, m, S at day 0 and day 50
z = n_series.shape[1] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

def _show(ax, arr, title, cmap):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

_show(axes[0, 0], n_series[0, z].detach().cpu().numpy(), "n (day 0)", "magma")
_show(axes[0, 1], m_series[0, z].detach().cpu().numpy(), "m (day 0)", "viridis")
_show(axes[0, 2], s_series[0, z].detach().cpu().numpy(), "S (day 0)", "plasma")

_show(axes[1, 0], n_series[-1, z].detach().cpu().numpy(), "n (day 50)", "magma")
_show(axes[1, 1], m_series[-1, z].detach().cpu().numpy(), "m (day 50)", "viridis")
_show(axes[1, 2], s_series[-1, z].detach().cpu().numpy(), "S (day 50)", "plasma")

plt.tight_layout()

## Notes

- This mini run is intended for **fast model sanity checks** only.
- For calibration/forecasting, tune parameters (`B`, `Dn`, `Ds`, `k_s`, transition parameters) and use patient-specific vascular masks if available.
- You can increase horizon and reduce `step_size` after the short run looks stable.

## Full run to last visit

This section mirrors `HGG_Demo` / `TNBC_Demo`: integrate from first to last visit with denser output.  
Then we compare the first 50 days of this full run against the mini-run.

In [ ]:
# Full run: first visit -> last visit (0.5-day output)
full_t0 = patient_data.visits[0].time
full_t1 = patient_data.visits[-1].time

full_timepoints = []
cur_t = full_t0
while cur_t <= full_t1:
    full_timepoints.append(cur_t)
    cur_t += timedelta(days=0.5)
if full_timepoints[-1] != full_t1:
    full_timepoints.append(full_t1)

times_full, traj_full = solver.solve(timepoints=full_timepoints, u_initial=u0)

n_full = traj_full[:, 0]
m_full = traj_full[:, 1]
s_full = traj_full[:, 2]

print("Full run complete")
print("  visits window (days):", (full_t1 - full_t0).days)
print("  output points:", len(full_timepoints))
print("  trajectory shape:", tuple(traj_full.shape))
print("  all finite:", bool(torch.isfinite(traj_full).all()))

In [ ]:
# Compare mini-run vs first 50 days of full-run
full_days = times_full.detach().cpu().numpy()
mini_days = times_mini.detach().cpu().numpy()

mask_50 = full_days <= 50.0
n_mass_full_50 = n_full[mask_50].sum(dim=(1, 2, 3)).detach().cpu().numpy()
m_mass_full_50 = m_full[mask_50].sum(dim=(1, 2, 3)).detach().cpu().numpy()
s_mean_full_50 = s_full[mask_50].mean(dim=(1, 2, 3)).detach().cpu().numpy()

n_mass_mini = n_series.sum(dim=(1, 2, 3)).detach().cpu().numpy()
m_mass_mini = m_series.sum(dim=(1, 2, 3)).detach().cpu().numpy()
s_mean_mini = s_series.mean(dim=(1, 2, 3)).detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(full_days[mask_50], n_mass_full_50, label="full run (<=50d)", alpha=0.9)
axes[0].plot(mini_days, n_mass_mini, "--", label="mini run 50d", alpha=0.9)
axes[0].set_title("Total n")
axes[0].set_xlabel("days")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(full_days[mask_50], m_mass_full_50, label="full run (<=50d)", alpha=0.9)
axes[1].plot(mini_days, m_mass_mini, "--", label="mini run 50d", alpha=0.9)
axes[1].set_title("Total m")
axes[1].set_xlabel("days")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(full_days[mask_50], s_mean_full_50, label="full run (<=50d)", alpha=0.9)
axes[2].plot(mini_days, s_mean_mini, "--", label="mini run 50d", alpha=0.9)
axes[2].set_title("Mean S")
axes[2].set_xlabel("days")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()

# Visual compare center slice at ~day 50
idx50_full = int(np.argmin(np.abs(full_days - 50.0)))
idx50_mini = int(np.argmin(np.abs(mini_days - 50.0)))
z = n_series.shape[1] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

def _imshow(ax, arr, title, cmap):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

_imshow(axes[0, 0], n_full[idx50_full, z].detach().cpu().numpy(), "n full ~day50", "magma")
_imshow(axes[0, 1], m_full[idx50_full, z].detach().cpu().numpy(), "m full ~day50", "viridis")
_imshow(axes[0, 2], s_full[idx50_full, z].detach().cpu().numpy(), "S full ~day50", "plasma")

_imshow(axes[1, 0], n_series[idx50_mini, z].detach().cpu().numpy(), "n mini day50", "magma")
_imshow(axes[1, 1], m_series[idx50_mini, z].detach().cpu().numpy(), "m mini day50", "viridis")
_imshow(axes[1, 2], s_series[idx50_mini, z].detach().cpu().numpy(), "S mini day50", "plasma")

plt.tight_layout()